# RecomendaAI - Treinamento do Modelo de Recomendação

Este notebook demonstra o processo de treinamento e avaliação dos modelos de recomendação para o projeto RecomendaAI.

Abordagens:
1. **Content-Based Filtering**: Baseado nos metadados dos filmes (gêneros e sinopse).
2. **Collaborative Filtering**: Baseado no comportamento do usuário (avaliações).

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split
import pickle

print("✅ Bibliotecas carregadas!")

## 1. Carregamento de Dados

In [ ]:
def load_movies(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

movies_data = load_movies('../data/tmdb_movies_large.json')
df_movies = pd.DataFrame(movies_data)
print(f"🎬 {len(df_movies)} filmes carregados.")
df_movies.head(3)

## 2. Content-Based Filtering (TF-IDF)
Vamos usar gêneros e sinopse para encontrar filmes similares.

In [ ]:
# Preparar os dados: combinar gêneros e sinopse em uma única string de texto
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))
df_movies['content'] = df_movies['genres_str'] + ' ' + df_movies['overview'].fillna('')

# Vetorização TF-IDF
tfidf = TfidfVectorizer(stop_words='portuguese')
tfidf_matrix = tfidf.fit_transform(df_movies['content'])

print(f"Matrix TF-IDF gerada: {tfidf_matrix.shape}")

# Calcular similaridade de cosseno (isso pode ser pesado para muitos dados, 
# mas para 1000 filmes é instantâneo)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def get_content_recommendations(title, n=10):
    if title not in df_movies['title'].values:
        return []
    
    idx = df_movies[df_movies['title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return df_movies.iloc[movie_indices][['id', 'title', 'vote_average']]

print("🎯 Exemplo de recomendação para 'Anônimo 2':")
print(get_content_recommendations('Anônimo 2', n=5))

## 3. Collaborative Filtering (SVD)
Como não temos um dataset de avaliações real, vamos gerar um sintético para demonstração.

In [ ]:
# Gerar ratings sintéticos: 100 usuários, cada um avaliando alguns filmes
np.random.seed(42)
num_users = 100
ratings_data = []
movie_ids = df_movies['id'].tolist()

for user_id in range(1, num_users + 1):
    num_user_ratings = np.random.randint(10, 50)
    chosen_movies = np.random.choice(movie_ids, num_user_ratings, replace=False)
    for movie_id in chosen_movies:
        rating = np.random.uniform(1, 5)
        ratings_data.append([user_id, movie_id, rating])

df_ratings = pd.DataFrame(ratings_data, columns=['user_id', 'item_id', 'rating'])
print(f"📊 {len(df_ratings)} avaliações sintéticas geradas.")

# Treinar modelo SVD
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_ratings[['user_id', 'item_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2)
algo = SVD()
algo.fit(trainset)

predictions = algo.test(testset)
print(f"📉 RMSE: {accuracy.rmse(predictions):.4f}")

# Exemplo de predição para o usuário 1 e o filme 1035259
pred = algo.predict(1, 1035259)
print(f"🔮 Predição para Usuário 1 no filme {pred.iid}: {pred.est:.2f}")

## 4. Exportação dos Modelos

In [ ]:
os.makedirs('../models/weights', exist_ok=True)

# Salvar o modelo SVD
with open('../models/weights/svd_model.pkl', 'wb') as f:
    pickle.dump(algo, f)

# Salvar a matriz TF-IDF e o vetorizador para o Content-Based
with open('../models/weights/tfidf_model.pkl', 'wb') as f:
    pickle.dump((tfidf, tfidf_matrix), f)

print("💾 Modelos salvos em models/weights/")